# Day 059 — Exercise 5: Poll Until Done

When a client submits a background job and gets a `job_id`, it needs a way to wait for completion. **Polling** repeatedly calls `GET /jobs/{id}` until the status leaves `pending`/`running`.

`poll_until_done` abstracts this into a reusable helper that takes a `status_fn` — any callable that returns a status dict for a job_id. This makes it testable with mock functions without a real server.

In [ ]:
import time
from typing import Callable


## Task

Implement `poll_until_done(status_fn, job_id, timeout=5.0, interval=0.05)`:

- Call `status_fn(job_id)` repeatedly
- If `status` is `'pending'` or `'running'`, sleep `interval` and try again
- If `status` is anything else (`'done'`, `'error'`, etc.), return the dict
- If `timeout` seconds elapse without completion, raise `TimeoutError`

## Your Implementation

In [ ]:
def poll_until_done(
    status_fn: Callable[[str], dict],
    job_id: str,
    timeout: float = 5.0,
    interval: float = 0.05,
) -> dict:
    """Poll status_fn(job_id) until status is not 'pending' or 'running'.

    status_fn: callable that takes job_id and returns a dict with a 'status' key
    Returns the final status dict (including 'result' if done).
    Raises TimeoutError if the job has not completed within `timeout` seconds.
    """
    # TODO: loop until done or timeout, sleep interval between polls
    raise NotImplementedError


In [ ]:
def poll_until_done(status_fn, job_id, timeout=5.0, interval=0.05):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        state = status_fn(job_id)
        if state.get("status") not in ("pending", "running"):
            return state
        time.sleep(interval)
    raise TimeoutError(f"Job {job_id!r} did not complete within {timeout}s")


## Automated checks

In [ ]:
score, total = 0, 4
try:
    # job done immediately
    immediate = lambda jid: {"status": "done", "result": 42}
    result = poll_until_done(immediate, "j1", timeout=1.0, interval=0.001)
    assert result == {"status": "done", "result": 42}, f"Got {result}"
    score += 1; print("\u2705 returns immediately when job already done")

    # job takes a few polls (pending → running → done)
    _state = {"phase": 0}
    def multi_phase(jid):
        _state["phase"] += 1
        if _state["phase"] == 1: return {"status": "pending"}
        if _state["phase"] == 2: return {"status": "running"}
        return {"status": "done", "result": "ready"}

    result2 = poll_until_done(multi_phase, "j2", timeout=1.0, interval=0.001)
    assert result2["status"] == "done"
    assert result2["result"] == "ready"
    score += 1; print("\u2705 polls through pending→running→done correctly")

    # error status is returned (not raised)
    error_fn = lambda jid: {"status": "error", "error": "boom"}
    result3 = poll_until_done(error_fn, "j3", timeout=1.0, interval=0.001)
    assert result3["status"] == "error"
    score += 1; print("\u2705 error status returned without raising")

    # timeout raises TimeoutError
    always_pending = lambda jid: {"status": "pending"}
    timed_out = False
    try:
        poll_until_done(always_pending, "j4", timeout=0.05, interval=0.001)
    except TimeoutError:
        timed_out = True
    assert timed_out, "Should have raised TimeoutError"
    score += 1; print("\u2705 raises TimeoutError after timeout exceeded")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def poll_until_done(status_fn, job_id, timeout=5.0, interval=0.05):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        state = status_fn(job_id)
        if state.get("status") not in ("pending", "running"):
            return state
        time.sleep(interval)
    raise TimeoutError(f"Job {job_id!r} did not complete within {timeout}s")
```

**`time.monotonic()`** is the right clock for measuring elapsed time — it never goes backwards (unlike `time.time()`, which can jump due to NTP adjustments). The polling loop returns the dict for any terminal status (`done`, `error`, `not_found`) — the caller decides what to do with errors.

</details>